In [1]:
!pip install GEOparse pandas numpy scikit-learn

In [2]:
"""Shows which metadata field carries the disease status (reads GEO cache, fast)."""
import re, collections, GEOparse
ACC = "GSE48350"
g = GEOparse.get_GEO(ACC, destdir="./geo_cache", silent=True)
norm = lambda v: re.sub(r"\d+", "#", v.lower())[:90]
fields = collections.defaultdict(collections.Counter)
for s in g.gsms.values():
    for k, v in s.metadata.items():
        if k in ("characteristics_ch1", "title"):
            for i, item in enumerate(v):
                fields[f"{k}[{i}]"][norm(item)] += 1
        else:
            fields[k][norm(" | ".join(v) if isinstance(v, list) else str(v))] += 1
for k, c in fields.items():
    if 1 < len(c) <= 12:                      # fields that actually vary, but not per-sample IDs
        print(f"\n--- {k} ({len(c)} distinct)")
        for val, n in c.most_common():
            print(f"  {n:>4} x {val}")


/usr/local/lib/python3.13/dist-packages/GEOparse/GEOparse.py:401: DtypeWarning: Columns (2: SPOT_ID) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, sep="\t")



--- status (3 distinct)
   172 x public on oct # #
    80 x public on apr # #
     1 x public on may # #

--- submission_date (3 distinct)
   251 x jun # #
     1 x sep # #
     1 x dec # #

--- last_update_date (2 distinct)
   246 x jun # #
     7 x may # #

--- characteristics_ch1[0] (5 distinct)
   138 x individual: #, c
    47 x gender: female
    33 x individual: #, aa
    33 x gender: male
     2 x individual: #-#, c

--- characteristics_ch1[1] (5 distinct)
    80 x age (yrs): #
    48 x brain region: superior frontal gyrus
    43 x brain region: postcentral gyrus
    43 x brain region: hippocampus
    39 x brain region: entorhinal cortex

--- characteristics_ch1[2] (6 distinct)
    91 x gender: male
    82 x gender: female
    25 x brain region: post-central gyrus
    21 x brain region: superior frontal gyrus
    19 x brain region: hippocampus
    15 x brain region: entorhinal cortex

--- characteristics_ch1[3] (9 distinct)
   173 x age (yrs): #
    23 x braak stage: vi
    19 

In [3]:
"""
merge_lodo_all_v4.py (EXPLORATORY extension of the previous run) -- ONE self-contained file (v2 loaders + v3 nested-CV and preprocessing variants).
Colab: paste the whole thing into ONE cell and run. Nothing to upload except your CSV
(preprocessed_expression.csv) which must already be in the Colab session or on Drive.

Pre-declared (before seeing results):
  PRIMARY = variant "Zf" (per-group z-score + label-free low-variance gene filter)
            + nested-CV tuning of k and C on the TRAINING data only.
  Z / R / C are ablations: ALL are reported, none is chosen from test results.
"""
import subprocess, sys, importlib, glob, os
for pkg, mod in [("GEOparse", "GEOparse"), ("inmoose", "inmoose")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import re
import warnings
import numpy as np
import pandas as pd
import GEOparse
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, confusion_matrix
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
REGION_KEYWORD = "hippocampus"
MIN_AGE = 69
SEED = 42
N_BOOT = 2000
KEEP_FRAC = 0.5
GRID = {"fs__k": [50, 100, 200, 500], "clf__C": [0.01, 0.1, 1.0]}
VARIANTS = ["Z", "Zf", "Zf25", "Zf75", "C", "CF", "QF", "R"]
REGIONS = ["entorhinal cortex", "hippocampus", "medial temporal gyrus",
           "posterior cingulate", "superior frontal gyrus", "primary visual cortex"]


def find_csv():
    for p in ["preprocessed_expression.csv", "/content/preprocessed_expression.csv",
              "/content/drive/MyDrive/preprocessed_expression.csv"]:
        if os.path.exists(p):
            return p
    hits = glob.glob("/content/**/preprocessed_expression*.csv", recursive=True)
    if hits:
        return hits[0]
    raise SystemExit("preprocessed_expression.csv not found. Put it in /content (or mount Drive) and rerun.")


# ---------------- loaders ----------------
def load_yours(path):
    d = pd.read_csv(path)
    d.columns = [c.strip() for c in d.columns]
    y = (d["Patient"] == "yes").astype(int).values
    X = d.drop(columns=["ID_REF", "Patient", "Stage"])
    X = X.T.groupby(level=0).mean().T
    return X, y


def geo_expression(gse):
    expr = gse.pivot_samples("VALUE")
    gpl = list(gse.gpls.values())[0].table
    sym = next(c for c in gpl.columns if c.lower() in ("gene symbol", "gene_symbol"))
    p2g = gpl.set_index("ID")[sym].dropna().astype(str).str.strip()
    p2g = p2g[(~p2g.str.contains("///")) & (p2g != "")]
    expr = expr.loc[expr.index.intersection(p2g.index)].copy()
    expr["gene"] = p2g.loc[expr.index].values
    expr["m"] = expr.drop(columns="gene").mean(axis=1)
    expr = expr.sort_values("m", ascending=False).drop_duplicates("gene")
    expr = expr.set_index("gene").drop(columns="m")
    X = expr.T
    if X.values.max() > 100:
        X = np.log2(X.clip(lower=1))
    return X


def _sample_text(s):
    parts = []
    for k in ("characteristics_ch1", "title", "source_name_ch1"):
        v = s.metadata.get(k, [])
        parts.append(" | ".join(v) if isinstance(v, list) else str(v))
    return " | ".join(parts).lower()


def geo_meta(gse, acc=""):
    rows = []
    for gsm, s in gse.gsms.items():
        text = _sample_text(s)
        if acc == "GSE48350":
            desc = " ".join(s.metadata.get("description", [])).lower()
            chars = " | ".join(s.metadata.get("characteristics_ch1", [])).lower()
            if "braak stage" in chars or "mmse" in chars or desc.strip() == "post-mortem tissue":
                lab = 1
            elif "cognitively intact" in desc:
                lab = 0
            else:
                lab = np.nan
        else:
            is_ad = re.search(r"alzheimer|\bad\b|affected", text)
            is_ctl = re.search(r"normal|control|non-?demented|\bnl\b", text)
            lab = 1 if (is_ad and not is_ctl) else 0 if (is_ctl and not is_ad) else np.nan
        m = re.search(r"age[^0-9]{0,15}(\d{2,3})", text)
        age = float(m.group(1)) if m else np.nan
        rows.append((gsm, lab, REGION_KEYWORD in text, age, text))
    return pd.DataFrame(rows, columns=["gsm", "label", "region_ok", "age", "text"]).set_index("gsm")


def load_geo(acc, min_age=None):
    gse = GEOparse.get_GEO(acc, destdir="./geo_cache", silent=True)
    X, meta = geo_expression(gse), geo_meta(gse, acc)
    print(f"\n=== CHECKPOINT {acc} ===")
    print("platform:", list(gse.gpls.keys()))
    print("labels (all):", meta["label"].value_counts(dropna=False).to_dict())
    print(f"region '{REGION_KEYWORD}':", int(meta["region_ok"].sum()), "samples")
    print("age parsed for", int(meta["age"].notna().sum()), "of", len(meta), "samples")
    keep = meta["region_ok"] & meta["label"].notna()
    if min_age is not None and meta["age"].notna().sum() > 0:
        keep &= meta["age"] >= min_age
    m = meta[keep]
    print("kept:", m["label"].value_counts().to_dict(),
          "| mean age by label:", m.groupby("label")["age"].mean().round(1).to_dict())
    if m["label"].nunique() < 2:
        print(f"!! {acc}: only one class after filtering -> dataset EXCLUDED.")
        return None
    return X.loc[m.index], m["label"].astype(int).values


def load_gse5281_all():
    gse = GEOparse.get_GEO("GSE5281", destdir="./geo_cache", silent=True)
    X, meta = geo_expression(gse), geo_meta(gse, "GSE5281")
    meta["region"] = meta["text"].apply(lambda t: next((r for r in REGIONS if r in t), None))
    m = meta[meta["label"].notna() & meta["region"].notna()]
    print("\n=== CHECKPOINT GSE5281 (all regions) ===")
    print(pd.crosstab(m["region"], m["label"]))
    print("mean age by label:", m.groupby("label")["age"].mean().round(1).to_dict())
    common = X.index.intersection(m.index)
    return X.loc[common], m.loc[common, "label"].astype(int).values, m.loc[common, "region"].values


# ---------------- preparation (all label-free) ----------------
def zscore_groups(X, groups):
    v = X.values.astype(float).copy()
    for g in np.unique(groups):
        i = groups == g
        mu, sd = v[i].mean(0), v[i].std(0, ddof=1)
        sd[~np.isfinite(sd) | (sd < 1e-8)] = 1.0
        v[i] = (v[i] - mu) / sd
    return pd.DataFrame(np.nan_to_num(v), index=X.index, columns=X.columns)


def prepare(raw):
    cols = None
    for n in raw:
        X = raw[n]["X"].copy()
        X.columns = pd.Index(X.columns).astype(str).str.strip().str.upper()
        raw[n]["X"] = X.T.groupby(level=0).mean().T
        cols = raw[n]["X"].columns if cols is None else cols.intersection(raw[n]["X"].columns)
    cols = pd.Index(sorted(cols)); print("\nShared genes:", len(cols))
    blocks = {n: dict(X=raw[n]["X"][cols], y=np.asarray(raw[n]["y"]), g=np.asarray(raw[n]["g"])) for n in raw}
    h = blocks["GSE5281_all"]; m = h["g"] == "hippocampus"
    blocks["GSE5281_hip"] = dict(X=h["X"][m], y=h["y"][m], g=h["g"][m])
    for b in blocks.values():
        b["Z"] = zscore_groups(b["X"], b["g"])
    return blocks


def combat(blocks, base):
    from inmoose.pycombat import pycombat_norm
    M = np.vstack([blocks[n]["X"].values for n in base])
    batch = np.concatenate([[f"{n}|{g}" for g in blocks[n]["g"]] for n in base])
    ok = np.ones(M.shape[1], bool)
    for bt in np.unique(batch):
        ok &= M[batch == bt].std(0) > 1e-6
    out = pycombat_norm(M[:, ok].T, batch).T
    cols = blocks[base[0]]["X"].columns[ok]
    res, i = {}, 0
    for n in base:
        k = len(blocks[n]["y"])
        res[n] = zscore_groups(pd.DataFrame(out[i:i + k], index=blocks[n]["X"].index, columns=cols), blocks[n]["g"])
        i += k
    res["GSE5281_hip"] = res["GSE5281_all"][blocks["GSE5281_all"]["g"] == "hippocampus"]
    return res


def qnorm(X, g):
    """Quantile normalisation across samples, done separately inside each group (label-free)."""
    v = X.values.astype(float).copy()
    for grp in np.unique(g):
        i = g == grp
        ref = np.sort(v[i], axis=1).mean(0)
        v[i] = ref[v[i].argsort(1).argsort(1)]
    return pd.DataFrame(v, index=X.index, columns=X.columns)


def add_variants(blocks):
    base = [n for n in blocks if n != "GSE5281_hip"]
    var = pd.concat([blocks[n]["X"].groupby(blocks[n]["g"]).var().mean(0) for n in base], axis=1).mean(1)
    top = lambda f: var[var >= var.quantile(1 - f)].index
    keep = top(KEEP_FRAC)
    C = combat(blocks, base)
    for n in blocks:
        b = blocks[n]
        b["Zf"] = b["Z"][keep]                                   # PRIMARY (pre-declared)
        b["Zf25"] = b["Z"][top(0.25)]                            # filter sensitivity
        b["Zf75"] = b["Z"][top(0.75)]                            # filter sensitivity
        b["C"] = C[n]
        b["CF"] = C[n][keep.intersection(C[n].columns)]          # ComBat + filter
        b["QF"] = zscore_groups(qnorm(b["X"], b["g"]), b["g"])[keep]   # quantile norm + filter
        b["R"] = zscore_groups(b["X"].rank(axis=1, pct=True), b["g"])
    print(f"Zf keeps {len(keep)} genes")


# ---------------- evaluation: tuning only on the training set ----------------
EXPS = [
    ("E1  GSE5281(all regions) -> yours         ", ["GSE5281_all"], "yours", False),
    ("E3  yours -> GSE5281(all regions)         ", ["yours"], "GSE5281_all", False),
    ("E3h yours -> GSE5281 hippocampus only     ", ["yours"], "GSE5281_hip", False),
    ("E2  GSE5281(all) -> GSE48350 hippo [CONF] ", ["GSE5281_all"], "GSE48350", True),
    ("E4  yours+GSE5281(all) -> GSE48350 [CONF] ", ["yours", "GSE5281_all"], "GSE48350", True),
]
PRIMARY = ("E1", "E3")


def run_nested(train, test, blocks, key):
    Xtr = np.vstack([blocks[n][key].values for n in train])
    ytr = np.concatenate([blocks[n]["y"] for n in train])
    cv = StratifiedKFold(min(5, np.bincount(ytr).min()), shuffle=True, random_state=SEED)
    pipe = Pipeline([("sc", StandardScaler()), ("fs", SelectKBest(f_classif)),
                     ("clf", LogisticRegression(max_iter=10000, solver="liblinear"))])
    gs = GridSearchCV(pipe, GRID, cv=cv, scoring="roc_auc").fit(Xtr, ytr)
    return blocks[test]["y"], gs.predict_proba(blocks[test][key].values)[:, 1], gs.best_params_


def boot_ci(y, p):
    r = np.random.default_rng(SEED); v = []
    for _ in range(N_BOOT):
        i = r.integers(0, len(y), len(y))
        if len(set(y[i])) == 2: v.append(roc_auc_score(y[i], p[i]))
    return (np.percentile(v, 2.5), np.percentile(v, 97.5)) if len(v) > 100 else (np.nan, np.nan)


def main():
    Xy, yy = load_yours(find_csv())
    Xg, yg, rg = load_gse5281_all()
    r48 = load_geo("GSE48350", min_age=MIN_AGE)
    raw = {"yours": dict(X=Xy, y=yy, g=np.array(["hippocampus"] * len(yy))),
           "GSE5281_all": dict(X=Xg, y=yg, g=rg)}
    if r48 is not None:
        raw["GSE48350"] = dict(X=r48[0], y=r48[1], g=np.array(["hippocampus"] * len(r48[1])))
    blocks = prepare(raw)
    add_variants(blocks)
    import joblib
    joblib.dump(list(blocks["yours"]["Zf"].columns), "model_genes.pkl")
    rows = []
    runnable = [e for e in EXPS if e[2] in blocks and all(t in blocks for t in e[1])]
    for var in VARIANTS:
        print(f"\n===== VARIANT {var}  (nested-CV tuning on training data only) =====")
        for label, tr, te, conf in runnable:
            y, p, bp = run_nested(tr, te, blocks, var)
            auc = roc_auc_score(y, p); lo, hi = boot_ci(y, p); pred = p >= 0.5
            tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
            ba = balanced_accuracy_score(y, pred)
            print(f"{label} n={len(y):>3} AUC={auc:.3f} ({lo:.3f}-{hi:.3f}) bal_acc={ba:.3f} "
                  f"sens={tp / max(tp + fn, 1):.2f} spec={tn / max(tn + fp, 1):.2f}  chosen={bp}")
            rows.append(dict(variant=var, exp=label.split()[0], conf=conf, n=len(y), auc=auc, ci_lo=lo, ci_hi=hi,
                             bal_acc=ba, sens=tp / max(tp + fn, 1), spec=tn / max(tn + fp, 1), **bp))
    df = pd.DataFrame(rows)
    df.to_csv("results_v3.csv", index=False)
    print("\n===== PRIMARY (E1,E3) mean AUC per variant (Zf = pre-declared primary) =====")
    print(df[df.exp.isin(PRIMARY)].groupby("variant")["auc"].mean().round(3).to_string())
    print("\n===== AUC / balanced-acc per variant and experiment (non-confounded) =====")
    nc = df[~df.conf]
    print(nc.pivot(index="variant", columns="exp", values="auc").round(3).to_string())
    print(nc.pivot(index="variant", columns="exp", values="bal_acc").round(3).to_string())
    print("\nSaved results_v3.csv. Report ALL variants; do not pick the best one from these test results.")


main()


=== CHECKPOINT GSE5281 (all regions) ===
label                    0   1
region                        
entorhinal cortex       13  10
hippocampus             13  10
medial temporal gyrus   12  16
posterior cingulate     13   9
primary visual cortex   12  19
superior frontal gyrus  11  23
mean age by label: {0: 79.5, 1: 79.8}

=== CHECKPOINT GSE48350 ===
platform: ['GPL570']
labels (all): {0: 173, 1: 80}
region 'hippocampus': 62 samples
age parsed for 253 of 253 samples
kept: {0: 24, 1: 18} | mean age by label: {0: 82.7, 1: 84.3}

Shared genes: 9152
Zf keeps 4576 genes

===== VARIANT Z  (nested-CV tuning on training data only) =====
E1  GSE5281(all regions) -> yours          n= 31 AUC=0.611 (0.386-0.813) bal_acc=0.528 sens=0.50 spec=0.56  chosen={'clf__C': 1.0, 'fs__k': 500}
E3  yours -> GSE5281(all regions)          n=161 AUC=0.738 (0.661-0.815) bal_acc=0.626 sens=0.79 spec=0.46  chosen={'clf__C': 1.0, 'fs__k': 50}
E3h yours -> GSE5281 hippocampus only      n= 23 AUC=0.769 (0.540-0.96

In [4]:
import joblib
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
import numpy as np

# 1. Re-run loaders locally to get 'blocks' globally
Xy, yy = load_yours(find_csv())
Xg, yg, rg = load_gse5281_all()
r48 = load_geo("GSE48350", min_age=MIN_AGE)
raw = {"yours": dict(X=Xy, y=yy, g=np.array(["hippocampus"] * len(yy))),
       "GSE5281_all": dict(X=Xg, y=yg, g=rg)}
if r48 is not None:
    raw["GSE48350"] = dict(X=r48[0], y=r48[1], g=np.array(["hippocampus"] * len(r48[1])))

blocks = prepare(raw)
add_variants(blocks)

# 2. Combine the training data from E1 & E3 as a robust final training set
X_final = np.vstack([blocks["GSE5281_all"]["Zf"].values, blocks["yours"]["Zf"].values])
y_final = np.concatenate([blocks["GSE5281_all"]["y"], blocks["yours"]["y"]])

# 3. Define the pipeline & parameter grid
pipe = Pipeline([
    ("sc", StandardScaler()),
    ("fs", SelectKBest(f_classif)),
    ("clf", LogisticRegression(max_iter=10000, solver="liblinear"))
])

cv = StratifiedKFold(min(5, np.bincount(y_final).min()), shuffle=True, random_state=SEED)
gs_final = GridSearchCV(pipe, GRID, cv=cv, scoring="roc_auc").fit(X_final, y_final)

# 4. Export the fitted GridSearchCV model to pkl
joblib.dump(gs_final, 'alzheimer_model.pkl')
print("Successfully saved final model to 'alzheimer_model.pkl'!")
print(f"Best cross-validated parameters: {gs_final.best_params_}")


=== CHECKPOINT GSE5281 (all regions) ===
label                    0   1
region                        
entorhinal cortex       13  10
hippocampus             13  10
medial temporal gyrus   12  16
posterior cingulate     13   9
primary visual cortex   12  19
superior frontal gyrus  11  23
mean age by label: {0: 79.5, 1: 79.8}

=== CHECKPOINT GSE48350 ===
platform: ['GPL570']
labels (all): {0: 173, 1: 80}
region 'hippocampus': 62 samples
age parsed for 253 of 253 samples
kept: {0: 24, 1: 18} | mean age by label: {0: 82.7, 1: 84.3}

Shared genes: 9152
Zf keeps 4576 genes
Successfully saved final model to 'alzheimer_model.pkl'!
Best cross-validated parameters: {'clf__C': 0.01, 'fs__k': 500}


In [6]:
!pip install -q streamlit joblib plotly pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 52.0 MB/s eta 0:00:00


In [10]:
!pip install -q streamlit


In [11]:
import streamlit as st
!pip install -q streamlit joblib plotly pillow

from google.colab import files
print("ارفع الأربعة مع بعض: alzheimer.py  ad_core.py  alzheimer_model.pkl  model_genes.pkl")
files.upload()

ارفع الأربعة مع بعض: alzheimer.py  ad_core.py  alzheimer_model.pkl  model_genes.pkl


Saving alzheimer.py to alzheimer.py
Saving ad_core.py to ad_core.py
Saving neurogene.py to neurogene.py
Saving alzheimer_model.pkl to alzheimer_model (1).pkl
Saving model_genes.pkl to model_genes (1).pkl


{'alzheimer.py': b'# -*- coding: utf-8 -*-\n"""NeuroGene - Alzheimer gene-expression classifier (HIERO 2026)."""\nimport html\nimport warnings\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nimport plotly.graph_objects as go\nimport streamlit as st\nfrom PIL import Image\nfrom sklearn.metrics import balanced_accuracy_score, roc_auc_score\n\nimport ad_core as core\n\nwarnings.filterwarnings("ignore")\nHERE = Path(__file__).parent\nINK, PAPER, SLIDE = "#16202B", "#F2F5F7", "#0D1520"\nTEAL, MAG = "#1FAA8C", "#D6336C"\n\nst.set_page_config(page_title="NeuroGene | Alzheimer expression classifier",\n                   page_icon="\xf0\x9f\xa7\xac", layout="wide")\n\nst.markdown("""\n<style>\n@import url(\'https://fonts.googleapis.com/css2?family=Source+Serif+4:opsz,wght@8..60,400;8..60,600&family=Schibsted+Grotesk:wght@400;500;600&display=swap\');\nhtml, body, .stApp, [class*="css"] { font-family:\'Schibsted Grotesk\', system-ui, sans-serif; color:#16202B;

In [12]:
!ls -la alzheimer.py ad_core.py alzheimer_model.pkl model_genes.pkl

-rw-r--r-- 1 root root   2393 Oct  8 20:50 ad_core.py
-rw-r--r-- 1 root root 192127 Oct  8 20:33 alzheimer_model.pkl
-rw-r--r-- 1 root root  14075 Oct  8 20:50 alzheimer.py
-rw-r--r-- 1 root root  38290 Oct  8 20:28 model_genes.pkl


In [13]:
import subprocess, sys, time, re, os

!pkill -f streamlit
!pkill -f cloudflared
if not os.path.exists("cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared

subprocess.Popen([sys.executable, "-m", "streamlit", "run", "alzheimer.py",
                  "--server.port", "8501", "--server.headless", "true",
                  "--server.enableCORS", "false", "--server.enableXsrfProtection", "false"],
                 stdout=open("streamlit.log", "w"), stderr=subprocess.STDOUT)
time.sleep(8)

subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
                 stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT)

url = None
for _ in range(30):
    time.sleep(2)
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("tunnel.log").read())
    if m:
        url = m.group(0)
        break
print("App URL:", url or "Not found, check tunnel.log")

رابط التطبيق: https://chips-rising-purse-calculators.trycloudflare.com
